# 🎬 HCL Movie Recommendation System
### Method: TF-IDF Feature Engineering + Cosine Similarity + K-Nearest Neighbors (KNN)
---

### 📌 Project Pipeline Architecture:
```
           10,000 MOVIE DATASET
                    │
                    ↓
          DATA PREPROCESSING & CLEANING
                    │
                    ↓
        FEATURE ENGINEERING (TAGS)
                    │
                    ↓
         TF-IDF VECTORIZATION (5,000 Features)
                    │
                    ↓
     COSINE SIMILARITY & KNN MODEL TRAINING
                    │
                    ↓
           MODEL TESTING (e.g. Electric Heart)
                    │
                    ↓
      EXPORT TO .pkl FILES FOR FLASK BACKEND
                    │
                    ↓
       FLASK API (/search & /recommend) ➔ FRONTEND
```


## 1. ⚙️ Library Imports
Importing essential machine learning and numerical libraries:
- `pandas`, `numpy` for data manipulation.
- `sklearn.feature_extraction.text.TfidfVectorizer` for NLP feature extraction.
- `sklearn.neighbors.NearestNeighbors` for K-Nearest Neighbors recommendation.
- `sklearn.metrics.pairwise.cosine_similarity` for similarity scoring.
- `pickle` for model serialization to `.pkl` format.


In [ ]:
import os
import sys
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

print("✅ Step 1: Libraries imported successfully!")


## 2. 📥 Load 10,000-Movie Dataset & Data Preprocessing
Loading the movie dataset and inspecting the metadata (title, genres, overview/tags).


In [ ]:
# Load the 10,000 movie dataset
# (If running on Colab, ensure movies_10k.csv is uploaded or downloaded)
DATASET_PATH = "movies_10k.csv"

# Fallback check for relative folder
if not os.path.exists(DATASET_PATH) and os.path.exists("HCL_Movie_Recommendation_Project/movies_10k.csv"):
    DATASET_PATH = "HCL_Movie_Recommendation_Project/movies_10k.csv"

movies_df = pd.read_csv(DATASET_PATH)

# Handle missing values
movies_df['overview'] = movies_df['overview'].fillna('')
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['clean_title'] = movies_df['clean_title'].fillna(movies_df['title'])

print(f"✅ Step 2: Loaded {len(movies_df):,} movies successfully!")
display(movies_df.head(5))


## 3. 🔤 Feature Engineering with TF-IDF Vectorizer
We transform textual information (genres, keywords, tags, overview) into high-dimensional numerical vectors using **TF-IDF (Term Frequency - Inverse Document Frequency)**:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$


In [ ]:
# Initialize TF-IDF Vectorizer with top 5,000 textual features
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit and transform movie overview / tags into TF-IDF vector matrix
tfidf_matrix = tfidf.fit_transform(movies_df['overview'])

print(f"✅ Step 3: TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Total vocabulary size: {len(tfidf.get_feature_names_out()):,} words")


## 4. 🧠 Model Training: K-Nearest Neighbors (KNN) & Cosine Similarity
Using `NearestNeighbors` with the **cosine** distance metric:
$$\text{Cosine Distance} = 1 - \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$$
This enables millisecond-fast neighbor retrieval across 10,000 items.


In [ ]:
# Train K-Nearest Neighbors with Cosine Distance
knn_model = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
knn_model.fit(tfidf_matrix)

# Compute full pairwise Cosine Similarity Matrix
similarity_matrix = cosine_similarity(tfidf_matrix)

print("✅ Step 4: KNN Model & Cosine Similarity Matrix trained successfully!")


## 5. 🎯 Recommendation Engine Function & Testing
Testing with the benchmark queries from yesterday's progress:
1. Query: **`Electric Heart`**
2. Query: **`Toy Story`**


In [ ]:
def get_movie_recommendations(movie_title, top_n=5):
    # Locate title in database
    match = movies_df[movies_df['clean_title'].str.contains(movie_title, case=False, na=False)]
    if match.empty:
        match = movies_df[movies_df['title'].str.contains(movie_title, case=False, na=False)]
        
    if match.empty:
        return f"❌ Movie '{movie_title}' not found in catalog."
    
    idx = match.index[0]
    matched_title = movies_df.iloc[idx]['title']
    matched_genres = movies_df.iloc[idx]['genres']
    
    # Query KNN for nearest neighbors
    distances, indices = knn_model.kneighbors(tfidf_matrix[idx], n_neighbors=top_n + 1)
    
    recs = []
    for i in range(1, len(indices[0])):
        m_idx = indices[0][i]
        dist = distances[0][i]
        match_score = f"{round((1 - dist) * 100, 1)}%"
        rec_row = movies_df.iloc[m_idx]
        recs.append({
            'Title': rec_row['title'],
            'Genres': rec_row['genres'],
            'Match Score': match_score
        })
        
    print(f"🎬 Recommendations for: '{matched_title}' [{matched_genres}]\n")
    return pd.DataFrame(recs)

# Test 1: Electric Heart
print("--- 🧪 Test 1: Electric Heart ---")
display(get_movie_recommendations("Electric Heart", top_n=5))

# Test 2: Toy Story
print("\n--- 🧪 Test 2: Toy Story ---")
display(get_movie_recommendations("Toy Story", top_n=5))


## 6. 💾 Save Trained Model Files (.pkl) for Flask Backend
Serializing the trained objects into `.pkl` files so the **Flask API** can load them instantly without retraining:
- `movies_list.pkl`: Complete DataFrame of 10,000 movies
- `movies_dict.pkl`: Dictionary representation for fast dictionary lookups
- `knn_model.pkl`: Trained NearestNeighbors model
- `similarity.pkl`: Cosine similarity matrix
- `tfidf_matrix.pkl`: Pre-computed TF-IDF embeddings


In [ ]:
# Export artifacts to .pkl
out_dir = "."

pickle.dump(movies_df, open(os.path.join(out_dir, "movies_list.pkl"), "wb"))
pickle.dump(movies_df.to_dict(), open(os.path.join(out_dir, "movies_dict.pkl"), "wb"))
pickle.dump(knn_model, open(os.path.join(out_dir, "knn_model.pkl"), "wb"))
pickle.dump(similarity_matrix, open(os.path.join(out_dir, "similarity.pkl"), "wb"))
pickle.dump(tfidf_matrix, open(os.path.join(out_dir, "tfidf_matrix.pkl"), "wb"))

print("✅ Step 6: All .pkl model files exported successfully!")
print("Files ready for Flask Backend: movies_list.pkl, movies_dict.pkl, knn_model.pkl, similarity.pkl, tfidf_matrix.pkl")


## 7. 🚀 Next Steps: Running the Flask Backend & Frontend
1. Keep the `.pkl` files in the same folder as `app.py`.
2. Run `python app.py` on your machine to start the Flask server on `http://127.0.0.1:5000`.
3. Open `frontend/index.html` in your browser to interact with the live autocomplete search and recommendations!
